# UPAR 2027 Track 2 — V3 Visual–Query Research

**Run all cells in Colab with a T4 GPU.** The notebook gets the experiment ZIP and your original checkpoint directly from **your own Google Drive**, downloads the official datasets if needed, trains the new visual–query heads, evaluates mADM, and **only builds a new Codabench ZIP if a held-out evaluation demonstrates improvement**.

Your existing V1 Codabench submission is never replaced. A 95% mADM score is a research goal, **not** a promised outcome.

**Before running:** Make sure your Drive contains `UPAR2027_Track2_CodaBenchAligned/UPAR_V3_VisualQuery_Research.zip` and the original `convnext_tiny_baseline/best_dev.pt`.


## 1. Mount Google Drive, extract research code

In [2]:

from google.colab import drive, files
from pathlib import Path
import os, sys, shutil, subprocess, zipfile, json

# 1. Mount Google Drive for trained model and results
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned')
CHECKPOINT = BASE / 'convnext_tiny_baseline/best_dev.pt'
PROJECT = Path('/content/UPAR-Challenge-2027')
WORK = Path('/content/upar_v3')
RESULTS = BASE / 'V3_VisualQuery_Results'

WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# 2. Upload research ZIP from your laptop
print("Select UPAR_V3_VisualQuery_Research.zip from your laptop")
uploaded = files.upload()

zip_files = [
    name for name in uploaded
    if name.lower().endswith('.zip')
]

assert len(zip_files) == 1, (
    "Please upload exactly one V3 research ZIP."
)

PACKAGE = Path('/content') / zip_files[0]

# 3. Verify and extract ZIP
assert zipfile.is_zipfile(PACKAGE), "Invalid ZIP file."

with zipfile.ZipFile(PACKAGE) as z:
    assert z.testzip() is None, "ZIP is corrupted."
    z.extractall(WORK)

# 4. Verify extracted research scripts
assert (WORK / 'train_visual_query.py').is_file(), (
    "train_visual_query.py was not found after extraction."
)

# 5. Verify existing ConvNeXt checkpoint
assert CHECKPOINT.is_file(), (
    f"ConvNeXt checkpoint missing: {CHECKPOINT}"
)

print("\nSUCCESS! V3 setup complete.")
print("Research package:", PACKAGE)
print("Extracted files:", WORK)
print("Original ConvNeXt checkpoint:", CHECKPOINT)
print("Results destination:", RESULTS)
print("\nYou can now run the next notebook cell.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Select UPAR_V3_VisualQuery_Research.zip from your laptop


Saving UPAR_V3_VisualQuery_Research.zip to UPAR_V3_VisualQuery_Research.zip

SUCCESS! V3 setup complete.
Research package: /content/UPAR_V3_VisualQuery_Research.zip
Extracted files: /content/upar_v3
Original ConvNeXt checkpoint: /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/convnext_tiny_baseline/best_dev.pt
Results destination: /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/V3_VisualQuery_Results

You can now run the next notebook cell.


## 2. Prepare official annotations and missing image datasets

In [3]:
# If your current Colab runtime already downloaded all 3 datasets, this won't download them again.
subprocess.run([sys.executable,'-m','pip','install','-q','gdown','tqdm','pandas','numpy','pillow'],check=True)
if not (PROJECT/'download_datasets.py').is_file():
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/speckean/UPAR-Challenge-2027.git',str(PROJECT)],check=True)

sys.path.insert(0,str(PROJECT))
import download_datasets
os.chdir(PROJECT)
DATA=PROJECT/'data'
expected={
    'Market1501':DATA/'Market1501/query/0001_c1s1_001051_00.jpg',
    'PA100K':DATA/'PA100k/release_data/release_data/090001.jpg',
    'PETA':DATA/'PETA/images/00001.png'
}
for name,file in expected.items():
    if file.is_file():
        print(name,'already downloaded')
        continue
    print('Downloading missing dataset:',name,'(large transfer, may take time)',flush=True)
    if name=='Market1501':download_datasets.prepare_market(DATA)
    elif name=='PA100K':download_datasets.prepare_pa100k(DATA)
    elif name=='PETA':download_datasets.prepare_peta(DATA)
    assert file.is_file(), f'Download/extraction incomplete for {name}: {file}'
print('All three example image paths exist. Verifying complete datasets next.')


Download Market 1501 dataset


Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=3fb6979d-832e-41e2-b691-91aab61d0efc
To: /content/UPAR-Challenge-2027/data/market_1501.zip
100%|██████████| 153M/153M [00:01<00:00, 118MB/s]


Extract Market 1501 dataset


Extracting : 100%|██████████| 68042/68042 [00:07<00:00, 9290.46it/s] 


Retrieving folder contents


...

Extract PA100k dataset


Extracting : 100%|██████████| 100001/100001 [00:12<00:00, 7720.89it/s]

Download PETA dataset
* url="https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1"
* destination="/content/UPAR-Challenge-2027/data/PETA/peta.zip"
...100%, 224 MB, 10477 KB/s, 21 seconds passed
Extract PETA dataset


Extracting : 100%|██████████| 19032/19032 [00:07<00:00, 2503.93it/s]
19010it [00:02, 8790.22it/s]

All three example image paths exist. Verifying complete datasets next.


## 3. Verify every training/validation image path and run unit tests

In [4]:
import torch
print('PyTorch version:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
assert torch.cuda.is_available(), 'A GPU is strongly recommended. Change Runtime -> GPU.'
subprocess.run([sys.executable,str(WORK/'check_dataset.py'),
                '--project-root',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(WORK/'tests'),'-v'],cwd=WORK,check=True)


PyTorch version: 2.11.0+cu130
CUDA available: True


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/upar_v3/tests', '-v'], returncode=0)

## 4. Train and evaluate V3 (takes time)

A one-time feature-cache pass through 97,669 training and 33,407 validation images is the costly part. The learning stage trains a lightweight dual-head model; early epochs should print fit-query mADM. The **holdout** is evaluated **only after training and selection have finished**. If Colab disconnects, rerun: feature caches and best heads are saved to Drive.

In [5]:
cmd=[sys.executable,'-u',str(WORK/'train_visual_query.py'),
     '--project-root',str(PROJECT),'--checkpoint',str(CHECKPOINT),
     '--output',str(RESULTS),'--epochs','8','--image-batch','96',
     '--batch','384','--workers','2','--fit-queries','192',
     '--holdout-queries','600']
print('Running:', ' '.join(cmd),flush=True)
with open(RESULTS/'run.log','a') as logfile:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='')
        logfile.write(line)
    rc=proc.wait()
assert rc==0, f'Training/evaluation failed (exit {rc}). Check {RESULTS / "run.log"}'
report=json.loads((RESULTS/'results.json').read_text())
print('Paired held-out result:',json.dumps(report['promotion'],indent=2))


Running: /usr/bin/python3 -u /content/upar_v3/train_visual_query.py --project-root /content/UPAR-Challenge-2027 --checkpoint /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/convnext_tiny_baseline/best_dev.pt --output /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/V3_VisualQuery_Results --epochs 8 --image-batch 96 --batch 384 --workers 2 --fit-queries 192 --holdout-queries 600
Device cuda Start Sun Oct 18 11:15:46 2026
Computing train features: 97669 images
  train: 96/97669
  train: 4896/97669
  train: 9696/97669
  train: 14496/97669
  train: 19296/97669
  train: 24096/97669
  train: 28896/97669
  train: 33696/97669
  train: 38496/97669
  train: 43296/97669
  train: 48096/97669
  train: 52896/97669
  train: 57696/97669
  train: 62496/97669
  train: 67296/97669
  train: 72096/97669
  train: 76896/97669
  train: 81696/97669
  train: 86496/97669
  train: 91296/97669
  train: 96096/97669
Computing val features: 33407 images
  val: 96/33407
  val: 4896/33407
  val: 9696/334

## 5. Promote only with evidence, build ZIP, then smoke-test it

In [6]:
from importlib.util import spec_from_file_location, module_from_spec
import numpy as np, tempfile
from PIL import Image

report=json.loads((RESULTS/'results.json').read_text())
verdict=report['promotion']
if not verdict['promote']:
    print('V3 DID NOT PASS PROMOTION GATE. Keep your V1 submission.')
    print('mADM delta:',verdict['delta_mADM'],'95% CI:',verdict['delta_ci95'])
else:
    candidate=RESULTS/'UPAR_Track2_V3_CANDIDATE.zip'
    subprocess.run([sys.executable,str(WORK/'build_submission.py'),
                    '--checkpoint',str(CHECKPOINT),
                    '--experiment-folder',str(RESULTS),
                    '--output-zip',str(candidate)],check=True)
    dest=Path('/content/upar_v3_submission_test')
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(candidate) as z:z.extractall(dest)
    spec=spec_from_file_location('upar_v3_run',str(dest/'run.py'))
    mod=module_from_spec(spec);spec.loader.exec_module(mod)
    mod.load_model()
    with tempfile.TemporaryDirectory() as td:
        tmp=Path(td)
        gal=[]
        for i in range(3):
            p=tmp/f'{i}.png'
            Image.fromarray(np.full((256,128,3),40+i*45,dtype=np.uint8)).save(p)
            gal.append({'index':i,'image':p.name,'image_path':str(p)})
        names=mod.CFG['attribute_names']
        q=np.zeros((2,40),dtype=int);q[0,1]=1
        r=mod.rank_gallery({'attribute_names':names,'queries':q.tolist(),'gallery':gal})
        mat=np.asarray(r['distances'])
        assert mat.shape==(2,3) and np.isfinite(mat).all()
    print('V3 CANDIDATE READY FOR OFFICIAL CODE SUBMISSION:',candidate)
    print('Official hidden-test result remains unknown. Do not delete V1.')


UPAR V3 model loaded cuda
Inference 3 / 3
V3 CANDIDATE READY FOR OFFICIAL CODE SUBMISSION: /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/V3_VisualQuery_Results/UPAR_Track2_V3_CANDIDATE.zip
Official hidden-test result remains unknown. Do not delete V1.


## Interpret results

- `V3_VisualQuery_Results/results.json`: baseline and V3 mADM/mAP/Rank1 on **disjoint holdout query identities** and a bootstrap uncertainty estimate.
- `V3_VisualQuery_Results/train_history.json`: experiment learning curves and selection history.
- If a V3 candidate ZIP is generated, that proves **local** improvement under our chosen gate. It **does not** prove hidden-set improvement or 95% mADM.
- No ZIP will be generated if Version 3 fails; keep and submit your original working **V1** ZIP before the challenge deadline.
